# 研究接口示例

`trader.research` 是界面之外的补充手段：所有返回值都是 pandas DataFrame，时间为纽约时间。
日常的回测、对比、复盘在研究版界面（`uv run trader-research`）里完成。

运行前请确认当前目录是项目根目录（或设置环境变量 `TRADER_HOME`）。

In [ ]:
import os
from pathlib import Path

# notebook 在 user/notebooks/ 下运行时，切回项目根目录
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parents[1])

from trader.research import (  # noqa: F401
    compute_indicator,
    load_bars,
    load_run,
    run_backtest,
    sweep,
)

## 1. 读取 bar
周期可以是 1s 5s 10s 30s 1m 2m 5m 15m 30m 1h 1d；`session="extended"` 包含盘前盘后。

In [ ]:
bars = load_bars("NVDA", "5m", "2026-09-28", "2026-10-02")
bars.head()

In [ ]:
# 每天常规时段的振幅
daily = bars.groupby(bars.index.date).agg(
    high=("high", "max"), low=("low", "min"), volume=("volume", "sum")
)
daily["range_pct"] = (daily["high"] / daily["low"] - 1) * 100
daily

## 2. 指标
内置指标和 `user/indicators/` 下的自定义指标都可以用，预热自动处理。

In [ ]:
vwap = compute_indicator("NVDA", "5m", "vwap", "2026-10-01", "2026-10-01")
orb = compute_indicator("NVDA", "5m", "opening_range", "2026-10-01", "2026-10-01", minutes=30)
vwap.join(orb.drop(columns="ts_end")).tail()

## 3. 回测
结果会保存到 `runs/`，并出现在界面的实验列表里。

In [ ]:
res = run_backtest(
    "orb_breakout",
    {"symbol": "NVDA", "notional": 15000},
    "2026-09-01",
    "2026-09-30",
    name="notebook 示例",
)
res

In [ ]:
res.summary["pnl"], res.summary["performance"]

In [ ]:
# 按入场原因和方向分组
res.trades.groupby(["entry_reason", "direction"])["net_pnl"].agg(["count", "sum", "mean"])

In [ ]:
# 在研究版界面中打开这次回测（界面没开就会启动它）
res.open_in_ui()

In [ ]:
# 以后可以按编号重新读取（编号在界面的实验列表里也能看到）
again = load_run(res.run_id)
again.summary["counts"]

## 4. 参数扫描
`split` 是样本外的第一天；样本内最好、样本外明显变差的参数组合多半是过拟合。

In [ ]:
table = sweep(
    "orb_breakout",
    {"symbol": "NVDA", "notional": 15000},
    {"minutes": [15, 30, 45]},
    "2026-08-03",
    "2026-09-30",
    split="2026-09-15",
)
table[["param_minutes", "in_net_pnl", "in_profit_factor", "out_net_pnl", "out_profit_factor"]]